In [1]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import spearmanr
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# Importiamo il mapper UNSUPERVISED
from ALIGNMENT_OFFICIAL_diff_dim_linear_unsupervised import StandaloneMapper

# optional SentenceTransformer backend
try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

# ===== Utilities =====
def load_embedding_backend(name):
    if name.startswith("sentence-transformers/") and SentenceTransformer is not None:
        model = SentenceTransformer(name)
        tok = model.tokenizer
        return tok, model, "sentencetransformer"
    else:
        tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        model = AutoModel.from_pretrained(name, trust_remote_code=True)
        return tok, model, "hf"

@torch.no_grad()
def embed_sentences(texts, tokenizer, model, device, batch_size=64, max_length=512):
    embeddings = []
    is_st = isinstance(model, SentenceTransformer)

    # Process in chunks to avoid OOM
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i+batch_size]
        if is_st:
            emb = model.encode(batch, convert_to_tensor=True, device=device, show_progress_bar=False)
            embeddings.append(emb.cpu())
        else:
            model = model.to(device).eval()
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
            out = model(**inputs)
            mask = inputs["attention_mask"].unsqueeze(-1)
            hidden = out.last_hidden_state * mask
            pooled = hidden.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu())

    return torch.cat(embeddings, dim=0)

# ===== Auto-Detect Architettura Mapper =====
def infer_mapper_arch_from_state(state):
    blocks_keys = [int(k.split(".")[1]) for k in state if k.startswith("blocks.")]
    n_layers = (1 + max(blocks_keys)) if blocks_keys else 4
    return {
        "input_dim" : state["input_proj.weight"].shape[1],
        "hidden_dim": state["input_proj.weight"].shape[0],
        "output_dim": state["output_proj.weight"].shape[0],
        "n_layers"  : n_layers,
    }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ===== Load models and mappers =====
src_name = "sentence-transformers/gtr-t5-base"
tgt_name = "intfloat/e5-base-v2"

# Assicurati di puntare alla cartella dei checkpoint UNSUPERVISED
path_mapper = "/home/pallucchini/NEW_ALIGNMENT_METHOD/align_new/saved_mappers_official_gtr_e5_1M_MacroStr_SimTlm_unsupervised"

src_tok, src_model, _ = load_embedding_backend(src_name)
tgt_tok, tgt_model, _ = load_embedding_backend(tgt_name)

# Carica SRC -> TGT
state_src2tgt = torch.load(f"{path_mapper}/official_mapper_src2tgt.pt", map_location=device)
arch_src2tgt = infer_mapper_arch_from_state(state_src2tgt)
mapper_src2tgt = StandaloneMapper(
    input_dim=arch_src2tgt["input_dim"],
    output_dim=arch_src2tgt["output_dim"],
    memory_dim=arch_src2tgt["output_dim"], # ignorato dal modulo, ma passiamo per parità
    hidden_dim=arch_src2tgt["hidden_dim"],
    n_layers=arch_src2tgt["n_layers"]
).to(device)
mapper_src2tgt.load_state_dict(state_src2tgt)
mapper_src2tgt = mapper_src2tgt.eval()

# Carica TGT -> SRC
state_tgt2src = torch.load(f"{path_mapper}/official_mapper_tgt2src.pt", map_location=device)
arch_tgt2src = infer_mapper_arch_from_state(state_tgt2src)
mapper_tgt2src = StandaloneMapper(
    input_dim=arch_tgt2src["input_dim"],
    output_dim=arch_tgt2src["output_dim"],
    memory_dim=arch_tgt2src["output_dim"], # ignorato dal modulo
    hidden_dim=arch_tgt2src["hidden_dim"],
    n_layers=arch_tgt2src["n_layers"]
).to(device)
mapper_tgt2src.load_state_dict(state_tgt2src)
mapper_tgt2src = mapper_tgt2src.eval()

/home/pallucchini/LLM_alignment/alignment_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0
Device: cuda


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16170.38it/s]


In [16]:
texts = ["gpt is a large language model developed by openai."]
# ===== Compute Source/Target Embeddings =====
src_embs = embed_sentences(texts, src_tok, src_model, device)
tgt_embs = embed_sentences(texts, tgt_tok, tgt_model, device)

direction = "src→tgt"

# ===== Apply Mapper THEN Normalize =====
if direction == "src→tgt":
    # 1. Map unnormalized embeddings (UNSUPERVISED: NO memory)
    mapped, _ = mapper_src2tgt(src_embs.to(device).float())
    
    # 2. Normalize AFTER mapping
    mapped = F.normalize(mapped, dim=-1).cpu()       
    
    # 3. Reference embeddings normalized
    ref_embs = F.normalize(tgt_embs, dim=-1)         
    original = src_embs
else:
    # 1. Map unnormalized embeddings (UNSUPERVISED: NO memory)
    mapped, _ = mapper_tgt2src(tgt_embs.to(device).float())
    
    mapped = F.normalize(mapped, dim=-1).cpu()       
    ref_embs = F.normalize(src_embs, dim=-1)         
    original = tgt_embs

sim = mapped @ ref_embs.T
print(sim)

Embedding: 100%|██████████| 1/1 [00:00<00:00, 126.65it/s]

tensor([[0.8409]], grad_fn=<MmBackward0>)
